# Imports


In [32]:
# Import required libraries
import polars as pl
from pathlib import Path

# Data Path


In [33]:
# Define data folder path
data_folder = Path("../data")

# Load Dataset


In [34]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview data
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Festival vs Non-Festival Sales


In [35]:
# Revenue comparison between festival and normal days
festival_sales = sales_df.group_by("festival").agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue"),

    pl.col("revenue").mean().round().alias("avg_order_value")
)

festival_sales.sort("total_revenue", descending=True)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22628\425732562.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,transactions,total_revenue,avg_order_value
str,u32,i64,f64
null,40087,13009013,325.0
"""Holi""",3331,1097119,329.0
"""Diwali""",3364,1088336,324.0
"""Summer""",3218,1051482,327.0


## Festival Revenue Share


In [36]:
festival_sales = festival_sales.with_columns(
    (
        pl.col("total_revenue") / pl.col("total_revenue").sum() * 100
    ).round(1).alias("revenue_share_pct")
)

festival_sales

festival,transactions,total_revenue,avg_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""Summer""",3218,1051482,327.0,6.5
"""Holi""",3331,1097119,329.0,6.8
"""Diwali""",3364,1088336,324.0,6.7
null,40087,13009013,325.0,80.1


## Festival Category Performance


In [37]:
festival_category = sales_df.group_by(
    ["festival", "category"]
).agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue")
).sort(
    ["festival", "total_revenue"],
    descending=[False, True]
)

festival_category

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22628\1494158052.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,category,transactions,total_revenue
str,str,u32,i64
null,"""Skincare""",24232,9848418
null,"""Haircare""",7849,1757101
null,"""Makeup""",8006,1403494
"""Diwali""","""Skincare""",2061,830189
"""Diwali""","""Haircare""",631,140369
…,…,…,…
"""Holi""","""Haircare""",676,152374
"""Holi""","""Makeup""",617,109233
"""Summer""","""Skincare""",1975,803225


# City Festival Performance


In [38]:
festival_city = sales_df.group_by(
    ["festival", "city"]
).agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue")
).sort(
    ["festival", "total_revenue"],
    descending=[False, True]
)

festival_city.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22628\820523211.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,city,transactions,total_revenue
str,str,u32,i64
null,"""South Dumdum""",355,114645
null,"""Allahabad""",354,111896
null,"""Meerut""",324,109526
null,"""Ghaziabad""",329,107821
null,"""Haldia""",326,105974
null,"""Berhampore""",322,105678
null,"""Nagercoil""",310,99590
null,"""Muzaffarpur""",287,93763
null,"""Uluberia""",284,93666


In [39]:
import polars as pl

# Aggregate sales by city
city_summary = (
    sales_df.group_by("city")
    .agg([
        pl.count("transaction_id").alias(
            "transactions"),  # total transactions per city
        # total revenue per city
        pl.sum("revenue").alias("total_revenue")
    ])
    .with_columns(
        # average revenue per transaction
        (pl.col("total_revenue") / pl.col("transactions")
         ).round().alias("avg_revenue_per_txn")
    )
)

city_summary.sort("avg_revenue_per_txn", descending=True)

city,transactions,total_revenue,avg_revenue_per_txn
str,u32,i64,f64
"""Bongaigaon""",38,14012,369.0
"""Motihari""",77,27873,362.0
"""Bhavnagar""",49,17751,362.0
"""New Delhi""",83,29767,359.0
"""Ajmer""",172,60978,355.0
…,…,…,…
"""Sangli-Miraj & Kupwad""",105,30695,292.0
"""Ludhiana""",50,14350,287.0
"""Khammam""",43,12307,286.0


## Save the Data set


In [40]:
festival_sales.write_parquet(data_folder/"festival_sales.parquet")

In [41]:
festival_sales

festival,transactions,total_revenue,avg_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""Summer""",3218,1051482,327.0,6.5
"""Holi""",3331,1097119,329.0,6.8
"""Diwali""",3364,1088336,324.0,6.7
null,40087,13009013,325.0,80.1
